# Analyses 1–2 — subdomains over time & path vs. filename

Assembled across all 15 federal domains. Reuses the deduplication + regex logic from the (archived) per-domain
notebooks; see `00_data_and_methods.ipynb` for method narrative.*

**Analyzes:** one unique URL per crawl year, deduped on
`(crawl_year, surtkey)`, `dns:` records. All counts below are therefore
deduped unique-URL counts.

Outputs are written to `analysis/em_deliverables/*.csv`.

- **Analysis 1 — subdomains over time:** counts per subdomain, per year, per
  domain, from `surthost_seg_2` (subdomain-1) and `surthost_seg_3` (subdomain-2).
- **Analysis 2 — path vs. filename+extension:** split the SURT path into a
  directory path vs. a trailing filename matching `.<2–4 chars>`; `.jpg` etc. are
  counted as files and excluded from the path side.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))   # repo root (this notebook lives in analysis/)
sys.path.insert(0, os.path.abspath('.'))
import config
import duckdb, pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_rows', 200); pd.set_option('display.width', 200)
YEAR_ORDER = ['2004','2008','2012','2016','2020','2024']
OUTDIR = 'em_deliverables'; os.makedirs(OUTDIR, exist_ok=True)

DBS = config.discover_domain_dbs('cdxj')
print(f"{len(DBS)}/15 domain DBs found:", ", ".join(DBS) or "(none — run on the server)")

def per_domain(sql):
    """Run `sql` against every domain DB; concat results with a `domain` column."""
    frames = []
    for dom, path in DBS.items():
        con = duckdb.connect(str(path), read_only=True)
        df = con.sql(sql).df(); con.close()
        df.insert(0, 'domain', dom); frames.append(df)
    out = pd.concat(frames, ignore_index=True)
    if 'crawl_year' in out.columns:
        out['crawl_year'] = out['crawl_year'].astype(str)
    return out

def year_cols(df):
    return [y for y in YEAR_ORDER if y in df.columns]

## Analysis 1 — number of subdomains over time, per domain

`surthost_seg_2` is subdomain-1 (e.g. `data` in `data.doi.gov`; SURT strips
`www`, so the apex is labeled `(bare/www)`). `surthost_seg_3` is subdomain-2
(e.g. `foo` in `foo.data.doi.gov`).

In [ ]:
SUBDOMAIN_SQL = r'''
WITH dedup AS (
  SELECT crawl_year, surtkey,
         min(surthost_seg_2) AS sub1,
         min(surthost_seg_3) AS sub2
  FROM eot_captures
  WHERE url NOT LIKE 'dns:%'
  GROUP BY 1, 2
)
SELECT COALESCE(sub1, '(bare/www)') AS subdomain_1,
       sub2                         AS subdomain_2,
       crawl_year,
       count(*)                     AS n
FROM dedup
GROUP BY 1, 2, 3
'''
subs = per_domain(SUBDOMAIN_SQL)        # domain, subdomain_1, subdomain_2, crawl_year, n
subs.to_csv(f'{OUTDIR}/a1_subdomains_long.csv', index=False)
print(f'{len(subs):,} (domain, subdomain_1, subdomain_2, year) rows')
subs.head()

### 1a. Headline — distinct subdomain-1 count per domain per year

"Number of subdomains over time for each gov domain." Pivot of the count of
distinct subdomain-1 labels (apex counted as one).

In [ ]:
distinct1 = (subs.groupby(['domain','crawl_year'])['subdomain_1'].nunique()
                 .reset_index(name='distinct_subdomains'))
d1_piv = distinct1.pivot(index='domain', columns='crawl_year', values='distinct_subdomains')
d1_piv = d1_piv[year_cols(d1_piv)].reindex([d for d in config.TARGET_DOMAINS if d in d1_piv.index])
d1_piv.to_csv(f'{OUTDIR}/a1_distinct_subdomain_counts.csv')
display(d1_piv.fillna(0).astype(int))

fig, ax = plt.subplots(figsize=(11,6))
for dom in d1_piv.index:
    r = d1_piv.loc[dom]
    ax.plot(r.index, r.values, 'o-', alpha=.6, label=dom)
ax.set_xlabel('crawl year'); ax.set_ylabel('distinct subdomain-1 labels')
ax.set_title('Number of distinct subdomains over time, by domain (deduplicated URLs)')
ax.legend(ncol=2, fontsize=8); plt.tight_layout()
plt.savefig(f'{OUTDIR}/a1_distinct_subdomains.png', dpi=150, bbox_inches='tight'); plt.show()

### 1b. Subdomain-1 URL counts per year, per domain (pivot tables)

The detailed deliverable: for each domain, a pivot of unique-URL counts per
subdomain-1 per year (e.g. `data.doi.gov` row = URLs under that subdomain each
year). One CSV per domain; doi shown inline as an example.

In [ ]:
sub1_counts = (subs.groupby(['domain','subdomain_1','crawl_year'])['n'].sum()
                   .reset_index())
for dom, g in sub1_counts.groupby('domain'):
    piv = g.pivot_table(index='subdomain_1', columns='crawl_year', values='n',
                        aggfunc='sum').fillna(0).astype(int)
    piv = piv[year_cols(piv)]
    piv['total'] = piv.sum(axis=1)
    piv = piv.sort_values('total', ascending=False)
    piv.to_csv(f"{OUTDIR}/a1_subdomain1_{dom.replace('.gov','')}.csv")
print('wrote per-domain subdomain-1 pivots to', OUTDIR)

# example: show one domain inline (doi if present, else the first)
example = 'doi.gov' if 'doi.gov' in DBS else list(DBS)[0]
g = sub1_counts[sub1_counts.domain==example]
ex_piv = g.pivot_table(index='subdomain_1', columns='crawl_year', values='n',
                       aggfunc='sum').fillna(0).astype(int)
ex_piv = ex_piv[year_cols(ex_piv)]; ex_piv['total'] = ex_piv.sum(axis=1)
print(f'Example — {example}: unique-URL counts per subdomain-1 per year')
ex_piv.sort_values('total', ascending=False).head(30)

### 1c. Subdomain-2 within subdomain-1 (`surthost_seg_3`)

Where a second subdomain level exists (e.g. `foo.data.doi.gov`). Long-form CSV
for all domains; top combinations shown inline.

In [ ]:
sub2 = subs[subs.subdomain_2.notna()].copy()
sub2_counts = (sub2.groupby(['domain','subdomain_1','subdomain_2','crawl_year'])['n']
                   .sum().reset_index())
sub2_counts.to_csv(f'{OUTDIR}/a1_subdomain2_long.csv', index=False)
if len(sub2_counts):
    piv2 = sub2_counts.pivot_table(index=['domain','subdomain_1','subdomain_2'],
                                   columns='crawl_year', values='n', aggfunc='sum').fillna(0).astype(int)
    piv2 = piv2[year_cols(piv2)]; piv2['total'] = piv2.sum(axis=1)
    display(piv2.sort_values('total', ascending=False).head(30))
else:
    print('No second-level subdomains found in the available DBs.')

## Analysis 2 — path vs. filename with extension

From the SURT path (query string excluded), a trailing segment matching
`.<2–4 alphanumerics>` is a **filename**; everything else is a **directory
path**. Extension-less URLs are path-only. Files like `.jpg` are counted in the
extension table and excluded from path analysis.

In [ ]:
PATHFILE_SQL = r'''
WITH dedup AS (
  SELECT crawl_year, surtkey,
         min(regexp_extract(surtkey, '\)([^?]*)', 1)) AS url_path
  FROM eot_captures
  WHERE url NOT LIKE 'dns:%'
  GROUP BY 1, 2
),
paths AS (
  SELECT crawl_year,
    NULLIF(regexp_extract(url_path, '/([^/]+\.[a-zA-Z0-9]{2,4})$', 1), '')      AS filename,
    NULLIF(lower(regexp_extract(url_path, '\.([a-zA-Z0-9]{2,4})$', 1)), '')      AS extension
  FROM dedup
)
SELECT crawl_year,
  count(*)                                                   AS unique_urls,
  sum(CASE WHEN filename IS NULL THEN 1 ELSE 0 END)          AS path_only,
  sum(CASE WHEN filename IS NOT NULL THEN 1 ELSE 0 END)      AS with_filename,
  round(100.0*sum(CASE WHEN filename IS NULL THEN 1 ELSE 0 END)/count(*),1) AS pct_path_only
FROM paths GROUP BY 1 ORDER BY 1
'''
split = per_domain(PATHFILE_SQL)
split.to_csv(f'{OUTDIR}/a2_path_vs_filename.csv', index=False)
split

### 2a. Path-only share over time (directory paths vs. files)

In [ ]:
piv = split.pivot(index='domain', columns='crawl_year', values='pct_path_only')
piv = piv[year_cols(piv)].reindex([d for d in config.TARGET_DOMAINS if d in piv.index])
plt.figure(figsize=(9,7))
sns.heatmap(piv, annot=True, fmt='.0f', cmap='Blues', vmin=0, vmax=100,
            cbar_kws={'label':'% URLs that are path-only (no filename)'}, linewidths=.5)
plt.title('Directory-path-only share of URLs, by domain and year')
plt.xlabel('crawl year'); plt.ylabel(''); plt.tight_layout()
plt.savefig(f'{OUTDIR}/a2_path_only_pct.png', dpi=150, bbox_inches='tight'); plt.show()

### 2b. Extension table (files only — `.jpg` etc., excluded from path analysis)

Per-year counts of file extensions, pooled across domains, plus a per-domain
long-form CSV.

In [ ]:
EXT_SQL = r'''
WITH dedup AS (
  SELECT crawl_year, surtkey,
         min(regexp_extract(surtkey, '\)([^?]*)', 1)) AS url_path
  FROM eot_captures WHERE url NOT LIKE 'dns:%' GROUP BY 1, 2
)
SELECT lower(regexp_extract(url_path, '\.([a-zA-Z0-9]{2,4})$', 1)) AS extension,
       crawl_year, count(*) AS n
FROM dedup
WHERE regexp_matches(url_path, '/[^/]+\.[a-zA-Z0-9]{2,4}$')
GROUP BY 1, 2
'''
ext = per_domain(EXT_SQL)
ext.to_csv(f'{OUTDIR}/a2_extensions_long.csv', index=False)

ext_piv = ext.pivot_table(index='extension', columns='crawl_year', values='n',
                          aggfunc='sum').fillna(0).astype(int)
ext_piv = ext_piv[year_cols(ext_piv)]; ext_piv['total'] = ext_piv.sum(axis=1)
ext_piv = ext_piv.sort_values('total', ascending=False)
ext_piv.to_csv(f'{OUTDIR}/a2_extensions_pivot.csv')
print('Top file extensions across all domains (deduplicated URLs):')
ext_piv.head(30)

## Deliverables written

All CSVs/PNGs are in `analysis/em_deliverables/`:

**Analysis 1 (subdomains)**
- `a1_distinct_subdomain_counts.csv` — distinct subdomain-1 count, domain × year (headline)
- `a1_subdomain1_<domain>.csv` — per-domain subdomain-1 URL-count pivots
- `a1_subdomains_long.csv`, `a1_subdomain2_long.csv` — full long-form

**Analysis 2 (path vs. filename)**
- `a2_path_vs_filename.csv` — path-only vs. with-filename split per domain-year
- `a2_extensions_pivot.csv`, `a2_extensions_long.csv` — extension tables (files only)
